
# Open-Meteo Hourly Forecast Pipeline

This notebook builds a simple medallion pipeline for hourly weather forecast data from the [Open-Meteo API](https://open-meteo.com).

## Pipeline Overview

1. **Bronze** — Raw hourly forecast snapshots are ingested into `workspace.open_meteo.bronze_hourly_forecast`. Each row is stamped with `fetched_at`, so the table keeps the full history of forecast snapshots (forecasts for the same hour can change between fetches).
2. **Silver** — `workspace.open_meteo.silver_hourly_forecast` keeps only the **latest forecast** per location and forecast hour, using `row_number()` over `(latitude, longitude, forecast_time)` ordered by `fetched_at DESC`.

## Key Columns

| Column | Description |
|---|---|
| `latitude`, `longitude` | Location of the forecast |
| `forecast_time` | The hour the forecast is for |
| `temperature_2m` / `apparent_temperature` | Actual and feels-like temperature |
| `precipitation` | Forecast precipitation |
| `wind_speed_10m` | Wind speed at 10m |
| `weather_code` | Open-Meteo weather code |
| `fetched_at` | Provenance — which snapshot the value came from |

## Usage Notes

- The silver table is rebuilt with `mode("overwrite")` on each run; re-run the cells in order.
- Deduplication keeps the most recent snapshot per `(latitude, longitude, forecast_time)`, so older forecast revisions are discarded from silver.


In [0]:
import logging

log = logging.getLogger("open_meteo.silver")

bronze_table = "workspace.open_meteo.bronze_hourly_forecast"
silver_table = "workspace.open_meteo.silver_hourly_forecast"
rejects_table = "workspace.open_meteo.silver_hourly_forecast_rejects"

EXPECTED_HOURS = 168  # 7 forecast days x 24 hours

IS_VALID = """
    COALESCE(
        temperature_2m BETWEEN -90 AND 60
        AND apparent_temperature BETWEEN -100 AND 70
        AND precipitation >= 0
        AND wind_speed_10m >= 0
        AND weather_code IS NOT NULL,
        false
    )
"""

spark.sql(f"""
    SELECT b.*
    FROM {bronze_table} b
    QUALIFY row_number() OVER (
        PARTITION BY b.latitude, b.longitude, b.forecast_time
        ORDER BY b.fetched_at DESC
    ) = 1
""").createOrReplaceTempView("deduped")

spark.sql(f"select * from deduped where {IS_VALID}").write.mode(
    "overwrite"
).saveAsTable(silver_table)

spark.sql(f"select * from deduped where not {IS_VALID}").write.mode(
    "overwrite"
).saveAsTable(rejects_table)

incomplete = spark.sql(f"""
    select latitude, longitude, fetched_at, count(*) as hours
    from {bronze_table}
    group by latitude, longitude, fetched_at
    having count(*) <> {EXPECTED_HOURS}
""")
if incomplete.count() > 0:
    log.warning("Cells with unexpected hour counts:\n%s", incomplete.toPandas())